# Bitcoin Sentiment vs Trader Performance Analysis

Using two datasets:
- fear_greed_index.csv - has the bitcoin fear and greed values daily
- historical_data.csv - has all the trades from hyperliquid

Goal is to see if market sentiment affects how traders perform


## importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

## loading the datasets

In [ ]:
fg = pd.read_csv('fear_greed_index.csv')
trades = pd.read_csv('historical_data.csv')

print("fear greed index shape:", fg.shape)
print(fg.head())

print("trades shape:", trades.shape)
print(trades.head())

## cleaning the data

In [ ]:
# fixing fear greed dates
fg['date'] = pd.to_datetime(fg['date'])
fg = fg[['date', 'value', 'classification']].copy()
fg.columns = ['date', 'fg_value', 'sentiment']
fg = fg.sort_values('date').reset_index(drop=True)

# fixing trades - the date column is like "02-12-2024 22:50"
trades['date'] = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
trades['date_only'] = pd.to_datetime(trades['date'].dt.date)

# renaming columns to make it easier
trades.rename(columns={
    'Account': 'account',
    'Coin': 'coin',
    'Execution Price': 'exec_price',
    'Size USD': 'size_usd',
    'Side': 'side',
    'Closed PnL': 'closed_pnl',
    'Fee': 'fee'
}, inplace=True)

trades['closed_pnl'] = pd.to_numeric(trades['closed_pnl'], errors='coerce').fillna(0)
trades['fee'] = pd.to_numeric(trades['fee'], errors='coerce').fillna(0)
trades['size_usd'] = pd.to_numeric(trades['size_usd'], errors='coerce').fillna(0)

print("trade dates from", trades['date_only'].min().date(), "to", trades['date_only'].max().date())
print("fear greed dates from", fg['date'].min().date(), "to", fg['date'].max().date())

## merging both datasets on date

In [ ]:
# joining the fear greed value to each trade based on the date
merged = trades.merge(fg, left_on='date_only', right_on='date', how='left')
merged = merged[merged['sentiment'].notna()].copy()

print("total rows after merge:", len(merged))
print()
print("trades per sentiment:")
print(merged['sentiment'].value_counts())

## average pnl by sentiment

In [ ]:
order = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']

pnl_by_sentiment = merged.groupby('sentiment')['closed_pnl'].agg(['mean', 'sum', 'count'])
pnl_by_sentiment.columns = ['avg_pnl', 'total_pnl', 'num_trades']
pnl_by_sentiment = pnl_by_sentiment.round(2)
pnl_by_sentiment = pnl_by_sentiment.reindex([x for x in order if x in pnl_by_sentiment.index])

print(pnl_by_sentiment)

## win rate by sentiment

In [ ]:
# a trade is a win if closed pnl is positive
merged['is_win'] = merged['closed_pnl'] > 0

win_rate = merged.groupby('sentiment')['is_win'].agg(['sum', 'count'])
win_rate.columns = ['wins', 'total_trades']
win_rate['win_rate_pct'] = (win_rate['wins'] / win_rate['total_trades'] * 100).round(1)
win_rate = win_rate.reindex([x for x in order if x in win_rate.index])

print(win_rate)

## buy vs sell performance under each sentiment

In [ ]:
side_pnl = merged.groupby(['sentiment', 'side'])['closed_pnl'].mean().unstack(fill_value=0).round(2)
side_pnl = side_pnl.reindex([x for x in order if x in side_pnl.index])

print(side_pnl)

## top 3 traders by total pnl in each sentiment

In [ ]:
top_traders = merged.groupby(['sentiment', 'account'])['closed_pnl'].sum().reset_index()
top_traders = top_traders.sort_values(['sentiment', 'closed_pnl'], ascending=[True, False])

for sent in order:
    temp = top_traders[top_traders['sentiment'] == sent].head(3)
    if not temp.empty:
        print(sent)
        print(temp[['account', 'closed_pnl']].to_string(index=False))
        print()

## correlation between daily pnl and fear greed index

In [ ]:
daily_pnl = merged.groupby('date_only')['closed_pnl'].sum().reset_index()
daily_pnl.columns = ['date_only', 'daily_pnl']
daily_pnl = daily_pnl.merge(fg, left_on='date_only', right_on='date', how='left')

corr = daily_pnl[['daily_pnl', 'fg_value']].corr().iloc[0, 1]
print("correlation between daily pnl and fear greed value:", round(corr, 3))

if corr > 0.2:
    print("positive correlation - traders make more money when market is greedy")
elif corr < -0.2:
    print("negative correlation - traders make more money when market is fearful")
else:
    print("weak correlation - fear greed index alone doesnt predict pnl well")

## plots

In [ ]:
colors = {
    'Extreme Fear': '#e74c3c',
    'Fear': '#e67e22',
    'Neutral': '#f1c40f',
    'Greed': '#2ecc71',
    'Extreme Greed': '#27ae60'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bitcoin Sentiment vs Trader Performance', fontsize=13, fontweight='bold')

bar_colors = [colors.get(s, 'gray') for s in pnl_by_sentiment.index]

# plot 1 - avg pnl by sentiment
ax1 = axes[0, 0]
bars = ax1.bar(pnl_by_sentiment.index, pnl_by_sentiment['avg_pnl'], color=bar_colors, edgecolor='black')
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_title('Average PnL by Sentiment')
ax1.set_ylabel('Avg PnL (USD)')
ax1.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, pnl_by_sentiment['avg_pnl']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha='center', va='bottom', fontsize=8)

# plot 2 - win rate by sentiment
ax2 = axes[0, 1]
ax2.bar(win_rate.index, win_rate['win_rate_pct'],
        color=[colors.get(s, 'gray') for s in win_rate.index], edgecolor='black')
ax2.axhline(50, color='navy', linewidth=1, linestyle='--', label='50%')
ax2.set_title('Win Rate by Sentiment')
ax2.set_ylabel('Win Rate (%)')
ax2.set_ylim(0, 100)
ax2.tick_params(axis='x', rotation=20)
ax2.legend(fontsize=8)

# plot 3 - buy vs sell
ax3 = axes[1, 0]
buy_col  = next((c for c in side_pnl.columns if 'BUY'  in str(c).upper()), None)
sell_col = next((c for c in side_pnl.columns if 'SELL' in str(c).upper()), None)
if buy_col and sell_col:
    x = np.arange(len(side_pnl))
    w = 0.35
    ax3.bar(x - w/2, side_pnl[buy_col],  w, label='BUY',  color='#3498db', edgecolor='black')
    ax3.bar(x + w/2, side_pnl[sell_col], w, label='SELL', color='#e74c3c', edgecolor='black')
    ax3.set_xticks(x)
    ax3.set_xticklabels(side_pnl.index, rotation=20)
    ax3.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax3.set_title('Buy vs Sell Avg PnL by Sentiment')
    ax3.set_ylabel('Avg PnL (USD)')
    ax3.legend()

# plot 4 - daily pnl vs fear greed over time
ax4 = axes[1, 1]
ax4b = ax4.twinx()
recent = daily_pnl.sort_values('date_only').tail(90)
ax4.bar(recent['date_only'], recent['daily_pnl'], color='#3498db', alpha=0.5, label='Daily PnL')
ax4b.plot(recent['date_only'], recent['fg_value'], color='orange', linewidth=1.5, label='FG Index')
ax4.set_title('Daily PnL vs Fear Greed Index (last 90 days)')
ax4.set_ylabel('Daily PnL (USD)', color='#3498db')
ax4b.set_ylabel('Fear Greed Value', color='orange')
ax4.tick_params(axis='x', rotation=30)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
ax4.xaxis.set_major_locator(mdates.MonthLocator())
lines1, l1 = ax4.get_legend_handles_labels()
lines2, l2 = ax4b.get_legend_handles_labels()
ax4.legend(lines1 + lines2, l1 + l2, fontsize=7)

plt.tight_layout()
plt.savefig('sentiment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("saved plot as sentiment_analysis.png")

## summary / key findings

In [ ]:
best = pnl_by_sentiment['avg_pnl'].idxmax()
worst = pnl_by_sentiment['avg_pnl'].idxmin()
best_wr = win_rate['win_rate_pct'].idxmax()

print("best avg pnl sentiment:", best, "->", pnl_by_sentiment.loc[best, 'avg_pnl'], "USD")
print("worst avg pnl sentiment:", worst, "->", pnl_by_sentiment.loc[worst, 'avg_pnl'], "USD")
print("highest win rate:", best_wr, "->", win_rate.loc[best_wr, 'win_rate_pct'], "%")
print("correlation pnl vs fear greed:", round(corr, 3))
print()
print("trade count per sentiment:")
for s in order:
    if s in pnl_by_sentiment.index:
        print(" ", s, ":", int(pnl_by_sentiment.loc[s, 'num_trades']), "trades")